In [8]:
from pyspark.sql import SparkSession 

In [2]:
spark = SparkSession.builder\
.appName("RDD_Operations")\
.getOrCreate()

26/04/18 15:48:28 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
customer_data = [
"customer_id,name,city,state,country,registration_date,is_active",
"0,Customer_0,Pune,Maharashtra,India,6/29/2023,FALSE",
"1,Customer_1,Bangalore,Tamil Nadu,India,12/7/2023,TRUE",
"2,Customer_2,Hyderabad,Gujarat,India,10/27/2023,TRUE",
"3,Customer_3,Bangalore,Karnataka,India,10/17/2023,FALSE",
"4,Customer_4,Ahmedabad,Karnataka,India,3/14/2023,FALSE",
"5,Customer_5,Hyderabad,Karnataka,India,7/28/2023,FALSE" ]


In [4]:
data_rdd = spark.sparkContext.parallelize(customer_data)

In [5]:
data_rdd.getNumPartitions()

2

## first() returns the first element of an RDD

In [6]:
header = data_rdd.first()

In [7]:
header

'customer_id,name,city,state,country,registration_date,is_active'

In [11]:
data_rdd

PythonRDD[2] at RDD at PythonRDD.scala:53

## Filter() -> based on the condition that is going to transform the dataset 

In [ ]:
data_rdd = data_rdd.filter(lambda row:row!=header)

In [12]:
data_rdd.collect()

['0,Customer_0,Pune,Maharashtra,India,6/29/2023,FALSE',
 '1,Customer_1,Bangalore,Tamil Nadu,India,12/7/2023,TRUE',
 '2,Customer_2,Hyderabad,Gujarat,India,10/27/2023,TRUE',
 '3,Customer_3,Bangalore,Karnataka,India,10/17/2023,FALSE',
 '4,Customer_4,Ahmedabad,Karnataka,India,3/14/2023,FALSE',
 '5,Customer_5,Hyderabad,Karnataka,India,7/28/2023,FALSE']

In [43]:
def parse_row(row):
    fields=row.split(',')
    return(
    int(fields[0]),
        fields[1],
        fields[2],
        fields[3],
        fields[4],
        fields[5],
        fields[6] == 'False'
    )

In [44]:
parsed_data = data_rdd.map(parse_row)

In [45]:
parsed_data.collect()

[(0, 'Customer_0', 'Pune', 'Maharashtra', 'India', '6/29/2023', False),
 (1, 'Customer_1', 'Bangalore', 'Tamil Nadu', 'India', '12/7/2023', False),
 (2, 'Customer_2', 'Hyderabad', 'Gujarat', 'India', '10/27/2023', False),
 (3, 'Customer_3', 'Bangalore', 'Karnataka', 'India', '10/17/2023', False),
 (4, 'Customer_4', 'Ahmedabad', 'Karnataka', 'India', '3/14/2023', False),
 (5, 'Customer_5', 'Hyderabad', 'Karnataka', 'India', '7/28/2023', False)]

## Advanced RDD Operations

Extract a field with Map() - Customer and State

In [17]:
name_city_rdd = parsed_data.map(lambda row : (row[1],row[2]))

In [18]:
name_city_rdd.collect()

[('Customer_0', 'Pune'),
 ('Customer_1', 'Bangalore'),
 ('Customer_2', 'Hyderabad'),
 ('Customer_3', 'Bangalore'),
 ('Customer_4', 'Ahmedabad'),
 ('Customer_5', 'Hyderabad')]

Filter out active customers

In [21]:
active_customers = parsed_data.filter(lambda row : row[6] == False)

In [22]:
active_customers.collect()

[(0, 'Customer_0', 'Pune', 'Maharashtra', 'India', '6/29/2023', False),
 (1, 'Customer_1', 'Bangalore', 'Tamil Nadu', 'India', '12/7/2023', False),
 (2, 'Customer_2', 'Hyderabad', 'Gujarat', 'India', '10/27/2023', False),
 (3, 'Customer_3', 'Bangalore', 'Karnataka', 'India', '10/17/2023', False),
 (4, 'Customer_4', 'Ahmedabad', 'Karnataka', 'India', '3/14/2023', False),
 (5, 'Customer_5', 'Hyderabad', 'Karnataka', 'India', '7/28/2023', False)]

# distinct () - Transformation

In [28]:
cities_rdd = parsed_data.map(lambda row:row[2]).distinct()

In [29]:
cities_rdd.collect()

['Pune', 'Hyderabad', 'Bangalore', 'Ahmedabad']

# take (n) gives the top n

In [30]:
cities_rdd.take(2)

['Pune', 'Hyderabad']

# ReduceByKey - Transformation
Combines values for each key using an associative reduce function 

In [31]:
customers_rdd = parsed_data.map(lambda row:(row[2],1)).reduceByKey(lambda x,y:x+y)

In [32]:
customers_rdd.collect()

[('Pune', 1), ('Hyderabad', 2), ('Bangalore', 2), ('Ahmedabad', 1)]

# CountByValue

In [36]:
cust_per_city = parsed_data.map(lambda row:row[2]).countByValue()

In [34]:
cust_per_city

defaultdict(int, {'Pune': 1, 'Bangalore': 2, 'Hyderabad': 2, 'Ahmedabad': 1})

In [ ]:
## reduceByKey is a transfrmation while countByValue is an action

## Combine More Operations

In [47]:
active_cities = parsed_data.map(lambda row:row[2])\
.distinct()

active_cities.collect()

['Pune', 'Hyderabad', 'Bangalore', 'Ahmedabad']

# Count Active Cities
# Run this code on 500MB file as well
# Include active cities check why it was not included 

In [48]:
active_cities.saveAsTextFile("/tmp/active_cities")

In [49]:
!hadoop fs -ls /tmp/

Found 4 items
drwxr-xr-x   - root          hadoop          0 2026-04-18 18:46 /tmp/active_cities
-rw-r--r--   2 mercy16samoei hadoop    1060750 2026-04-18 15:19 /tmp/customers.csv
drwxrwxrwt   - hdfs          hadoop          0 2026-03-17 10:41 /tmp/hadoop-yarn
drwx-wx-wx   - hive          hadoop          0 2026-03-17 10:41 /tmp/hive
